In [1]:
print("Spark session is working")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: d1846306-8414-4cde-bcdd-610f7883649c
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session d1846306-8414-4cde-bcdd-610f7883649c to get into ready status...
Session d1846306-8414-4cde-bcdd-610f7883649c has been created.
Spark session is working


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [3]:
BUCKET = "aws-ecommerce-data-wael-2026"

RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"
PROCESSED_PATH = f"s3://{BUCKET}/curated/ecommerce"


In [4]:
# 2. Read all raw CSV files
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/customers.csv")
)

categories = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/categories.csv")
)

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/products.csv")
)

departments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/departments.csv")
)

employees = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/employees.csv")
)

suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/suppliers.csv")
)

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/orders.csv")
)

order_details = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/order_details.csv")
)

payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/payments.csv")
)

product_suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/product_suppliers.csv")
)

shippers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shippers.csv")
)

shipments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shipments.csv")
)

In [5]:
print("Customers:", customers.count())
print("Categories:", categories.count())
print("Products:", products.count())
print("Departments:", departments.count())
print("Employees:", employees.count())
print("Suppliers:", suppliers.count())
print("Orders:", orders.count())
print("Order Details:", order_details.count())
print("Payments:", payments.count())
print("Product Suppliers:", product_suppliers.count())
print("Shippers:", shippers.count())
print("Shipments:", shipments.count())

Customers: 10000
Categories: 20
Products: 1000
Departments: 10
Employees: 200
Suppliers: 100
Orders: 50000
Order Details: 100000
Payments: 45000
Product Suppliers: 2027
Shippers: 10
Shipments: 40000


In [6]:
customers.printSchema()
orders.printSchema()
order_details.printSchema()
products.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

root
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Status: string (nullable = true)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: integer (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true

In [7]:
categories.printSchema()
departments.printSchema()
employees.printSchema()
payments.printSchema()
product_suppliers.printSchema()
shippers.printSchema()
shipments.printSchema()
suppliers.printSchema()

root
 |-- CategoryID: integer (nullable = true)
 |-- CategoryName: string (nullable = true)

root
 |-- DepartmentID: integer (nullable = true)
 |-- DepartmentName: string (nullable = true)

root
 |-- EmployeeID: integer (nullable = true)
 |-- ManagerID: integer (nullable = true)
 |-- DepartmentID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- HireDate: timestamp (nullable = true)

root
 |-- PaymentID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- PaymentDate: timestamp (nullable = true)
 |-- Amount: double (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- SupplierID: integer (nullable = true)

root
 |-- ShipperID: integer (nullable = true)
 |-- CompanyName: string (nullable = true)

root
 |-- ShipmentID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ShipperID: integer (

In [8]:
customers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in customers.columns
]).show()

+----------+---------+--------+-----+----+-------+----------------+
|CustomerID|FirstName|LastName|Email|City|Country|RegistrationDate|
+----------+---------+--------+-----+----+-------+----------------+
|         0|        0|       0|    0|   0|      0|               0|
+----------+---------+--------+-----+----+-------+----------------+


In [9]:
customers.show(10, truncate=False)
orders.show(10, truncate=False)

+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City      |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza      |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai     |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza      |United Kingdom|2020-12-22 00:00:00|
|4         |Patricia |Miller  |patricia.miller4@example.com |London    |Germany       |2020-05-10 00:00:00|
|5         |Robert   |Johnson |robert.johnson5@example.com  |Cairo     |Saudi Arabia  |2022-06-14 00:00:00|
|6         |Jeffery  |Wagner  |jeffery.wagner6@example.com  |Dubai     |United Kingdom|2020-04-18 00:00:00|
|7         |Anthony  |Gonzal

In [10]:
categories.show(10)
departments.show(10)
employees.show(10)
payments.show(10)
product_suppliers.show(10)
shippers.show(10)
shipments.show(10)
suppliers.show(10)
products.show(10)
order_details.show(10)

+----------+---------------+
|CategoryID|   CategoryName|
+----------+---------------+
|         1|    Electronics|
|         2|      Computers|
|         3|  Mobile Phones|
|         4|Home Appliances|
|         5|      Furniture|
|         6|       Clothing|
|         7|          Shoes|
|         8|         Sports|
|         9|          Books|
|        10|         Beauty|
+----------+---------------+
only showing top 10 rows

+------------+----------------+
|DepartmentID|  DepartmentName|
+------------+----------------+
|           1|              IT|
|           2|         Finance|
|           3|           Sales|
|           4|       Marketing|
|           5|              HR|
|           6|      Operations|
|           7|     Procurement|
|           8|Customer Service|
|           9|       Logistics|
|          10|      Management|
+------------+----------------+

+----------+---------+------------+---------+--------+-------+-------------------+
|EmployeeID|ManagerID|DepartmentID|F

In [11]:
customers.groupBy("customerid").count().filter(F.col("count") > 1).show()
orders.groupBy("orderid").count().filter(F.col("count") > 1).show()

+----------+-----+
|customerid|count|
+----------+-----+
+----------+-----+

+-------+-----+
|orderid|count|
+-------+-----+
+-------+-----+


In [12]:
def clean_column_names(df):
    for column in df.columns:
        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        
        df = df.withColumnRenamed(column, new_name)
    
    return df

In [13]:
customers = clean_column_names(customers)
categories = clean_column_names(categories)
products = clean_column_names(products)
departments = clean_column_names(departments)
employees = clean_column_names(employees)
suppliers = clean_column_names(suppliers)
orders = clean_column_names(orders)
order_details = clean_column_names(order_details)
payments = clean_column_names(payments)
product_suppliers = clean_column_names(product_suppliers)
shippers = clean_column_names(shippers)
shipments = clean_column_names(shipments)

In [14]:
customers.printSchema()

root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: timestamp (nullable = true)


In [15]:
departments.printSchema()

root
 |-- departmentid: integer (nullable = true)
 |-- departmentname: string (nullable = true)


In [16]:
def trim_all_strings(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
    return df

categories = trim_all_strings(categories)
products = trim_all_strings(products)
departments = trim_all_strings(departments)
employees = trim_all_strings(employees)
suppliers = trim_all_strings(suppliers)
order_details = trim_all_strings(order_details)
payments = trim_all_strings(payments)
product_suppliers = trim_all_strings(product_suppliers)
shippers = trim_all_strings(shippers)
shipments = trim_all_strings(shipments)

In [17]:
categories = categories.dropna(subset=["categoryid"])
products = products.dropna(subset=["productid"])
departments = departments.dropna(subset=["departmentid"])
employees = employees.dropna(subset=["employeeid"])
suppliers = suppliers.dropna(subset=["supplierid"])
payments = payments.dropna(subset=["paymentid"])
shippers = shippers.dropna(subset=["shipperid"])
shipments = shipments.dropna(subset=["shipmentid"])

In [18]:
customers = customers.dropDuplicates(["customerid"])

orders = orders.dropDuplicates(["orderid"])

products = products.dropDuplicates(["productid"])

order_details = order_details.dropDuplicates(["orderdetailid"])

In [19]:
customers = (
    customers
    .withColumn("firstname", F.trim("firstname"))
    .withColumn("lastname", F.trim("lastname"))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("city", F.trim("city"))
    .withColumn("country", F.trim("country"))
)

In [20]:
orders = (
    orders
    .withColumn("status", F.upper(F.trim("status")))
)

In [21]:
orders = (
    orders
    .withColumn(
        "orderid",
        F.col("orderid").cast("long")
    )
    .withColumn(
        "customerid",
        F.col("customerid").cast("long")
    )
    .withColumn(
        "orderdate",
        F.to_date("orderdate")
    )
)

In [22]:
customers.filter(~F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")).show(truncate=False)

orders.filter(~F.col("status").isin("PENDING", "SHIPPED", "DELIVERED", "CANCELLED")).show(truncate=False)

+----------+---------+--------+-----+----+-------+----------------+
|customerid|firstname|lastname|email|city|country|registrationdate|
+----------+---------+--------+-----+----+-------+----------------+
+----------+---------+--------+-----+----+-------+----------------+

+-------+----------+----------+----------+
|orderid|customerid|orderdate |status    |
+-------+----------+----------+----------+
|12     |2908      |2024-12-25|PROCESSING|
|38     |2252      |2025-08-15|PROCESSING|
|69     |257       |2024-11-03|PROCESSING|
|77     |1862      |2024-08-07|PROCESSING|
|93     |6380      |2025-09-29|PROCESSING|
|94     |2361      |2025-11-10|PROCESSING|
|138    |9458      |2025-08-18|PROCESSING|
|180    |8180      |2025-05-13|PROCESSING|
|193    |9710      |2025-11-18|PROCESSING|
|195    |7118      |2026-02-09|PROCESSING|
|211    |3133      |2024-11-22|PROCESSING|
|231    |1652      |2024-07-19|PROCESSING|
|249    |3256      |2026-06-23|PROCESSING|
|261    |8160      |2026-07-01|PROCESSI

In [23]:
customers = customers.dropna(subset=["customerid"])
orders = orders.dropna(subset=["orderid"])
products.filter(F.col("price").isNull() | (F.col("price") <= 0)).show()

+---------+----------+-----------+-----+-----+----+-----+
|productid|categoryid|productname|brand|price|cost|stock|
+---------+----------+-----------+-----+-----+----+-----+
+---------+----------+-----------+-----+-----+----+-----+


In [24]:
def cast_ids(df):
    for col_name in df.columns:
        if col_name.endswith("id"):
            df = df.withColumn(col_name, F.col(col_name).cast("long"))
    return df

categories = cast_ids(categories)
products = cast_ids(products)
departments = cast_ids(departments)
employees = cast_ids(employees)
suppliers = cast_ids(suppliers)
order_details = cast_ids(order_details)
payments = cast_ids(payments)
product_suppliers = cast_ids(product_suppliers)
shippers = cast_ids(shippers)
shipments = cast_ids(shipments)

In [25]:
products = products.withColumn("price", F.col("price").cast("decimal(12,2)"))
products = products.withColumn("cost", F.col("cost").cast("decimal(12,2)"))
order_details = order_details.withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)"))
payments = payments.withColumn("amount", F.col("amount").cast("decimal(12,2)"))
order_details = order_details.withColumn("quantity", F.col("quantity").cast("integer"))

In [26]:
order_details = order_details.withColumn(
    "quantity",
    F.when(
        F.col("quantity") < 0,
        0
    ).otherwise(F.col("quantity"))
)

In [27]:
order_details = order_details.withColumn(
    "unitprice",
    F.when(
        F.col("unitprice") < 0,
        0
    ).otherwise(F.col("unitprice"))
)

In [28]:
order_details = order_details.withColumn(
    "total_amount",
    F.col("quantity") * F.col("unitprice")
)

In [29]:
orders = (
    orders
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
    .withColumn("day", F.dayofmonth("orderdate"))
    .withColumn("quarter", F.quarter("orderdate"))
    .withColumn("day_of_week", F.dayofweek("orderdate"))
)

In [30]:
customers = customers.withColumn(
    "full_name", 
    F.concat_ws(" ", F.col("firstname"), F.col("lastname"))
)
customers.select("customerid", "firstname", "lastname", "full_name").show(5)

+----------+---------+--------+----------------+
|customerid|firstname|lastname|       full_name|
+----------+---------+--------+----------------+
|         1| Danielle| Johnson|Danielle Johnson|
|         2|   Joshua|  Walker|   Joshua Walker|
|         3|     Jill|  Rhodes|     Jill Rhodes|
|         4| Patricia|  Miller| Patricia Miller|
|         5|   Robert| Johnson|  Robert Johnson|
+----------+---------+--------+----------------+
only showing top 5 rows


In [31]:
order_sales = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
)

In [32]:
order_sales = order_sales.select(
    F.col("o.orderid"),
    F.col("o.customerid"),
    F.col("o.orderdate"),
    F.col("o.status"),
    F.col("od.productid"),
    F.col("od.quantity"),
    F.col("od.unitprice"),
    F.col("od.total_amount")
)

In [33]:
order_summary = order_details.groupBy("orderid").agg(
    F.sum("total_amount").alias("order_total_amount"),
    F.sum("quantity").alias("order_total_quantity")
)

In [34]:
order_summary = order_summary.withColumn(
    "order_value_category",
    F.when(F.col("order_total_amount") >= 1000, "HIGH")
    .when((F.col("order_total_amount") >= 500) & (F.col("order_total_amount") < 1000), "MEDIUM")
    .otherwise("LOW")
)

In [35]:
customer_sales_total = orders.join(order_summary, "orderid").groupBy("customerid").agg(F.sum("order_total_amount").alias("total_customer_sales"))

customer_sales_total = customer_sales_total.withColumn(
    "customer_segment",
    F.when(F.col("total_customer_sales") > 10000, "VIP")
    .when(F.col("total_customer_sales") > 5000, "PREMIUM")
    .when(F.col("total_customer_sales") > 1000, "REGULAR")
    .otherwise("LOW_VALUE")
)

In [36]:
products.select(F.avg("price").alias("average_product_price")).show()

+---------------------+
|average_product_price|
+---------------------+
|          1530.095400|
+---------------------+


In [37]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define the window
customer_window = (
    Window
    .partitionBy("customerid")
    .orderBy(F.col("orderdate").desc())
)

# Add row number
last_order_customer = (
    orders
    .withColumn(
        "last_order_rank",
        F.row_number().over(customer_window)
    )
)

# Keep only the latest order for each customer
last_order_customer = (
    last_order_customer
    .filter(F.col("last_order_rank") == 1)
)

last_order_customer.show()

+-------+----------+----------+----------+----+-----+---+-------+-----------+---------------+
|orderid|customerid| orderdate|    status|year|month|day|quarter|day_of_week|last_order_rank|
+-------+----------+----------+----------+----+-----+---+-------+-----------+---------------+
|  17011|         3|2026-04-01|   PENDING|2026|    4|  1|      2|          4|              1|
|   2861|         9|2025-11-29| DELIVERED|2025|   11| 29|      4|          7|              1|
|  33028|        11|2026-01-18|PROCESSING|2026|    1| 18|      1|          1|              1|
|  39870|        14|2026-03-13|   PENDING|2026|    3| 13|      1|          6|              1|
|  11899|        17|2026-06-07| DELIVERED|2026|    6|  7|      2|          1|              1|
|  11271|        28|2025-12-05| CANCELLED|2025|   12|  5|      4|          6|              1|
|  36170|        29|2026-03-20| CANCELLED|2026|    3| 20|      1|          6|              1|
|  27776|        30|2026-08-20|   SHIPPED|2026|    8| 20|   

In [48]:
first_order_window = Window.partitionBy("customerid").orderBy(F.col("orderdate").asc())
first_order_customer = orders.withColumn("first_order_rank", F.row_number().over(first_order_window)).filter(F.col("first_order_rank") == 1)
first_order_customer.show()

+-------+----------+----------+----------+----+-----+---+-------+-----------+----------------+
|orderid|customerid| orderdate|    status|year|month|day|quarter|day_of_week|first_order_rank|
+-------+----------+----------+----------+----+-----+---+-------+-----------+----------------+
|  12675|         3|2024-01-08| CANCELLED|2024|    1|  8|      1|          2|               1|
|  30999|         9|2024-02-28| DELIVERED|2024|    2| 28|      1|          4|               1|
|  26574|        11|2025-01-20| CANCELLED|2025|    1| 20|      1|          2|               1|
|  49268|        14|2024-02-24| DELIVERED|2024|    2| 24|      1|          7|               1|
|  47651|        17|2024-04-21|   SHIPPED|2024|    4| 21|      2|          1|               1|
|  48673|        28|2024-10-30|   PENDING|2024|   10| 30|      4|          4|               1|
|  33997|        29|2024-04-17| DELIVERED|2024|    4| 17|      2|          4|               1|
|  41614|        30|2024-02-09|PROCESSING|2024|   

In [49]:
sales_rank_window = Window.orderBy(F.col("total_customer_sales").desc())
ranked_customers = customer_sales_total.withColumn("sales_rank", F.rank().over(sales_rank_window))
ranked_customers.show()

+----------+--------------------+----------------+----------+
|customerid|total_customer_sales|customer_segment|sales_rank|
+----------+--------------------+----------------+----------+
|      6772|           368984.90|             VIP|         1|
|       245|           338329.93|             VIP|         2|
|      9895|           330803.47|             VIP|         3|
|      5446|           321600.74|             VIP|         4|
|      5688|           319630.49|             VIP|         5|
|      4971|           314924.38|             VIP|         6|
|      7013|           311641.81|             VIP|         7|
|      8502|           310361.58|             VIP|         8|
|      3556|           309252.08|             VIP|         9|
|       849|           307160.04|             VIP|        10|
|      6894|           303524.49|             VIP|        11|
|      7501|           303480.53|             VIP|        12|
|      1986|           301686.70|             VIP|        13|
|       

In [50]:
category_window = Window.partitionBy("categoryid").orderBy(F.col("price").desc())
ranked_products = products.withColumn("price_rank", F.row_number().over(category_window))
top_3_products = ranked_products.filter(F.col("price_rank") <= 3)
most_expensive_product = ranked_products.filter(F.col("price_rank") == 1)
top_3_products.show()
most_expensive_product.show()

+---------+----------+--------------------+---------+-------+-------+-----+----------+
|productid|categoryid|         productname|    brand|  price|   cost|stock|price_rank|
+---------+----------+--------------------+---------+-------+-------+-----+----------+
|      759|         1|      Bose Mouse 759|     Bose|2908.72|1474.59|  465|         1|
|      845|         1|Microsoft Headpho...|Microsoft|2880.76|2429.34|   85|         2|
|      564|         1|Microsoft Headpho...|Microsoft|2879.09|1663.19|  393|         3|
|      927|         2|   LG Television 927|       LG|2898.94|2272.73|   14|         1|
|      276|         2|  Samsung Camera 276|  Samsung|2865.65|2239.68|  447|         2|
|      753|         2|  LG Smart Watch 753|       LG|2864.85|2285.55|  156|         3|
|      247|         3|Samsung Televisio...|  Samsung|2872.79|1847.47|  110|         1|
|      742|         3|Lenovo Headphones...|   Lenovo|2829.74|1800.67|  223|         2|
|      978|         3|      HP Monitor 978|

In [51]:
shipment_window = Window.partitionBy("orderid").orderBy(F.col("shipdate").desc())
latest_shipment = shipments.withColumn("shipment_rank", F.row_number().over(shipment_window)).filter(F.col("shipment_rank") == 1)
latest_shipment.show()

+----------+-------+---------+-------------------+-------------------+-------------+
|shipmentid|orderid|shipperid|           shipdate|       deliverydate|shipment_rank|
+----------+-------+---------+-------------------+-------------------+-------------+
|     23754|      3|        6|2024-11-19 00:00:00|2024-11-21 00:00:00|            1|
|     27536|      4|        9|2025-04-10 00:00:00|2025-04-16 00:00:00|            1|
|     18580|      6|        3|2026-07-31 00:00:00|2026-08-06 00:00:00|            1|
|     11890|      9|        6|2026-05-05 00:00:00|2026-05-14 00:00:00|            1|
|     24881|     17|        8|2024-09-28 00:00:00|2024-09-29 00:00:00|            1|
|      2390|     19|        7|2026-06-09 00:00:00|2026-06-19 00:00:00|            1|
|     35951|     20|        1|2025-11-26 00:00:00|2025-12-04 00:00:00|            1|
|     27376|     23|        7|2024-06-14 00:00:00|2024-06-23 00:00:00|            1|
|     20302|     24|        5|2026-08-21 00:00:00|2026-08-22 00:0

In [52]:
orders_customers = orders.join(customers, "customerid", "left").select(
    "orderid", "customerid", F.col("full_name").alias("customer_name"),
    "city", "country", "orderdate", "status"
)

sales_dataset = orders.join(order_details, "orderid", "inner").join(products, "productid", "inner")
products_categories = products.join(categories, "categoryid", "left")
products_suppliers_full = products.join(product_suppliers, "productid", "left").join(suppliers, "supplierid", "left")
orders_payments = orders.join(payments, "orderid", "left")
orders_shipments = orders.join(shipments, "orderid", "left").join(shippers, "shipperid", "left")
orders_customers.show()
sales_dataset.show()

+-------+----------+----------------+----------+--------------+----------+----------+
|orderid|customerid|   customer_name|      city|       country| orderdate|    status|
+-------+----------+----------------+----------+--------------+----------+----------+
|      1|      8870|    Derrick King|Alexandria|United Kingdom|2024-05-03| CANCELLED|
|      3|      8030|     Eric Ortega|    Jeddah|       Germany|2026-01-14| DELIVERED|
|      5|      4623|     Laura Perry| Abu Dhabi|  Saudi Arabia|2025-05-26| DELIVERED|
|     10|      9903|      Robin Diaz|      Giza|United Kingdom|2025-08-15| CANCELLED|
|     12|      2908|    Tyler Golden|     Cairo| United States|2024-12-25|PROCESSING|
|     16|      2017|     David Smith|     Amman|        Kuwait|2024-06-01| DELIVERED|
|     18|      1459|  Nicole Vasquez|     Amman|United Kingdom|2025-08-05| DELIVERED|
|     27|      3858|    Sally Murphy|     Amman|           UAE|2024-12-03|   SHIPPED|
|     31|      8044|       Lisa York|    Jeddah|      

In [53]:
total_sales = order_details.select(F.sum("total_amount").alias("total_sales"))
total_orders = orders.select(F.count("orderid").alias("total_orders"))
total_qty_sold = order_details.select(F.sum("quantity").alias("total_quantity"))
avg_order_val = order_summary.select(F.avg("order_total_amount").alias("avg_order_value"))

sales_by_customer = sales_dataset.groupBy("customerid").agg(F.sum("total_amount").alias("total_sales"))
sales_by_product = sales_dataset.groupBy("productid").agg(F.sum("total_amount").alias("total_sales"))
sales_by_category = sales_dataset.groupBy("categoryid").agg(F.sum("total_amount").alias("total_sales"))

sales_by_country = orders_customers.join(order_details, "orderid", "inner").groupBy("country").agg(F.sum("total_amount").alias("total_sales"))
sales_by_city = orders_customers.join(order_details, "orderid", "inner").groupBy("city").agg(F.sum("total_amount").alias("total_sales"))

monthly_sales_agg = orders.join(order_details, "orderid", "inner").groupBy("year", "month").agg(F.sum("total_amount").alias("total_sales"))
yearly_sales = orders.join(order_details, "orderid", "inner").groupBy("year").agg(F.sum("total_amount").alias("total_sales"))
sales_by_status = orders.join(order_details, "orderid", "inner").groupBy("status").agg(F.sum("total_amount").alias("total_sales"))

orders_per_customer = orders.groupBy("customerid").count().withColumnRenamed("count", "num_orders")
total_sales.show()
sales_by_status.show()
orders_per_customer.show()

+------------+
| total_sales|
+------------+
|840507631.75|
+------------+

+----------+------------+
|    status| total_sales|
+----------+------------+
| DELIVERED|169408962.56|
|PROCESSING|168737103.72|
|   PENDING|168528735.38|
| CANCELLED|167507249.29|
|   SHIPPED|166325580.80|
+----------+------------+

+----------+----------+
|customerid|num_orders|
+----------+----------+
|      1027|         5|
|      9925|         7|
|      6072|         4|
|      5986|         5|
|      2307|        10|
|      8829|         7|
|      2460|         8|
|      7747|         3|
|      5337|         8|
|      8579|         7|
|      2887|         6|
|      5307|         5|
|      6447|         5|
|      1004|         3|
|      2127|         6|
|      1990|         7|
|      3491|         9|
|      9434|         4|
|      7160|         6|
|      8936|         6|
+----------+----------+
only showing top 20 rows


In [54]:
top_10_customers = sales_by_customer.orderBy(F.col("total_sales").desc()).limit(10)
top_10_products = sales_by_product.orderBy(F.col("total_sales").desc()).limit(10)
top_5_categories = sales_by_category.orderBy(F.col("total_sales").desc()).limit(5)

never_ordered_customers = customers.join(orders, "customerid", "left_anti")
never_ordered_products = products.join(order_details, "productid", "left_anti")
more_than_10_orders = orders_per_customer.filter(F.col("num_orders") > 10)

mismatched_payments = order_summary.join(payments, "orderid", "inner").filter(F.col("order_total_amount") != F.col("amount"))
orders_no_shipment = orders.join(shipments, "orderid", "left_anti")
shipments_no_order = shipments.join(orders, "orderid", "left_anti")
top_10_customers.show()
top_10_products.show()
top_5_categories.show()

+----------+-----------+
|customerid|total_sales|
+----------+-----------+
|      6772|  368984.90|
|       245|  338329.93|
|      9895|  330803.47|
|      5446|  321600.74|
|      5688|  319630.49|
|      4971|  314924.38|
|      7013|  311641.81|
|      8502|  310361.58|
|      3556|  309252.08|
|       849|  307160.04|
+----------+-----------+

+---------+-----------+
|productid|total_sales|
+---------+-----------+
|      582| 2128226.94|
|      794| 2037242.52|
|      650| 1954709.20|
|       71| 1915462.35|
|      911| 1858710.64|
|      232| 1857950.64|
|      191| 1841616.40|
|      125| 1830229.92|
|      335| 1826738.13|
|      350| 1823336.58|
+---------+-----------+

+----------+-----------+
|categoryid|total_sales|
+----------+-----------+
|        12|57306656.35|
|         1|56420102.49|
|         6|52495546.95|
|        19|49331847.38|
|         5|48833064.16|
+----------+-----------+


In [55]:
daily_sales = orders.join(order_details, "orderid").groupBy("orderdate").agg(F.sum("total_amount").alias("daily_sales"))
quarterly_sales = orders.join(order_details, "orderid").groupBy("year", "quarter").agg(F.sum("total_amount").alias("quarterly_sales"))

highest_sales_month = monthly_sales_agg.orderBy(F.col("total_sales").desc()).limit(1)
highest_orders_day = orders.groupBy("orderdate").count().orderBy(F.col("count").desc()).limit(1)
avg_orders_per_month = orders.groupBy("year", "month").count().agg(F.avg("count").alias("avg_orders_per_month"))
highest_sales_month.show()
highest_orders_day.show()
avg_orders_per_month.show()

+----+-----+-----------+
|year|month|total_sales|
+----+-----+-----------+
|2025|    3|27889327.64|
+----+-----+-----------+

+----------+-----+
| orderdate|count|
+----------+-----+
|2024-09-10|   75|
+----------+-----+

+--------------------+
|avg_orders_per_month|
+--------------------+
|  1515.1515151515152|
+--------------------+


In [47]:
customers.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/customers/")
products.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/products/")
categories.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/categories/")
order_details.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/order_details/")
payments.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/payments/")
shipments.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/shipments/")

orders.write.mode("overwrite").partitionBy("year", "month").parquet(f"{PROCESSED_PATH}/orders/")

sales_dataset.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/fact_sales/")
sales_by_customer.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/customer_sales/")
sales_by_product.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/product_sales/")
sales_by_category.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/category_sales/")
monthly_sales_agg.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/monthly_sales/")

In [56]:
spark.stop()